# 15 Thesis Figures and Tables

Το notebook παράγει τα τελικά σχήματα και τους πίνακες που χρησιμοποιούνται στην παρουσίαση και στη συγγραφή της διπλωματικής εργασίας.


In [ ]:
# =============================================================================
# Εισαγωγές και καθολικές ρυθμίσεις γραφημάτων
# =============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# --- Κατάλογος εξόδου για τα σχήματα ---
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# --- Ύφος γραφημάτων ---
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
})

# --- Χρωματική παλέτα pipelines ---
PALETTE = {
    'dense': '#4C72B0',           # Μπλε
    'hybrid': '#55A868',          # Πράσινο
    'hybrid_reranked': '#C44E52', # Κόκκινο
}
LABELS = {
    'dense': 'Dense (Baseline)',
    'hybrid': 'Hybrid',
    'hybrid_reranked': 'Hybrid + Reranking',
}
PIPELINES = ['dense', 'hybrid', 'hybrid_reranked']

print('Οι ρυθμίσεις γραφημάτων φορτώθηκαν.')
print(f'Τα σχήματα θα αποθηκευτούν στο: ./{FIG_DIR}/')


In [ ]:
# =============================================================================
# Φόρτωση δεδομένων
# =============================================================================

# --- Retrieval Evaluation ---
df_retrieval = pd.read_csv('../data/processed/evaluation/retrieval_evaluation_comparison.csv')

# --- QA Evaluation ---
df_qa = pd.read_csv('../data/processed/evaluation/qa_evaluation_comparison.csv')

# --- RAGChecker Summaries ---
def load_ragchecker_summary(path, run_name):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lstrip('\ufeff')
    df['run_name'] = run_name
    return df

rc_dense    = load_ragchecker_summary('../data/processed/evaluation/ragchecker_summary_dense.csv', 'dense')
rc_hybrid   = load_ragchecker_summary('../data/processed/evaluation/ragchecker_summary_hybrid.csv', 'hybrid')
rc_reranked = load_ragchecker_summary('../data/processed/evaluation/ragchecker_summary_reranked.csv', 'hybrid_reranked')
df_rc = pd.concat([rc_dense, rc_hybrid, rc_reranked], ignore_index=True)

# Pivot to wide format
df_rc_wide = df_rc.pivot_table(
    index='run_name', columns='metric_name', values='metric_value'
).reset_index()

# --- RAGAS ---
df_ragas = pd.read_csv('../data/processed/evaluation/ragas_results.csv')

# --- RAGChecker Details (per-query) ---
def load_rc_details(path, run_name):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lstrip('\ufeff')
    df['run_name'] = run_name
    return df

rc_det_dense    = load_rc_details('../data/processed/evaluation/ragchecker_details_dense.csv', 'dense')
rc_det_hybrid   = load_rc_details('../data/processed/evaluation/ragchecker_details_hybrid.csv', 'hybrid')
rc_det_reranked = load_rc_details('../data/processed/evaluation/ragchecker_details_reranked.csv', 'hybrid_reranked')
df_rc_details   = pd.concat([rc_det_dense, rc_det_hybrid, rc_det_reranked], ignore_index=True)

# --- FinanceBench Master ---
df_master = pd.read_csv('../data/processed/evaluation/financebench_master_results.csv')

print('Τα σύνολα δεδομένων φορτώθηκαν.')
print(f'   Retrieval evaluation: {df_retrieval.shape}')
print(f'   QA evaluation: {df_qa.shape}')
print(f'   RAGChecker (all pipelines): {df_rc_details.shape}')
print(f'   RAGAS results: {df_ragas.shape}')
df_rc_wide


In [ ]:
# =============================================================================
# Σχήμα 1: Απόδοση retrieval με Hit@k και MRR
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 1: Retrieval Performance Across Pipeline Configurations',
             fontsize=14, fontweight='bold', y=1.02)

# --- Panel A: Hit@k grouped bar chart ---
ax = axes[0]
metrics_hitk = ['evidence_hit_at_1', 'evidence_hit_at_3', 'evidence_hit_at_5']
labels_hitk  = ['Hit@1', 'Hit@3', 'Hit@5']

x = np.arange(len(metrics_hitk))
width = 0.25

for i, pipeline in enumerate(PIPELINES):
    row = df_retrieval[df_retrieval['run_name'] == pipeline].iloc[0]
    values = [row[m] * 100 for m in metrics_hitk]
    bars = ax.bar(x + i * width, values, width,
                  label=LABELS[pipeline], color=PALETTE[pipeline],
                  alpha=0.88, edgecolor='white', linewidth=0.5)
    # Value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(labels_hitk)
ax.set_ylabel('Hit Rate (%)')
ax.set_title('(A) Evidence Hit Rate at k')
ax.set_ylim(0, 100)
ax.legend(loc='upper left', framealpha=0.9)

# --- Panel B: MRR bar chart ---
ax2 = axes[1]
mrr_values = [df_retrieval[df_retrieval['run_name'] == p].iloc[0]['mrr'] * 100
              for p in PIPELINES]
bars = ax2.bar([LABELS[p] for p in PIPELINES], mrr_values,
               color=[PALETTE[p] for p in PIPELINES],
               alpha=0.88, edgecolor='white', linewidth=0.5, width=0.5)
for bar, val in zip(bars, mrr_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.set_ylabel('MRR (%)')
ax2.set_title('(B) Mean Reciprocal Rank (MRR)')
ax2.set_ylim(0, 85)
ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig1_retrieval_performance.png')
plt.show()
print('Figure 1 saved.')


In [ ]:
# =============================================================================
# Σχήμα 2: Μετρικές retriever και generator από το RAGChecker
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 2: RAGChecker Fine-Grained Evaluation',
             fontsize=14, fontweight='bold', y=1.02)

# --- Panel A: Retriever metrics ---
ax = axes[0]
retriever_metrics = ['claim_recall', 'context_precision']
ret_labels = ['Claim Recall', 'Context Precision']

x = np.arange(len(retriever_metrics))
width = 0.25

for i, pipeline in enumerate(PIPELINES):
    row = df_rc_wide[df_rc_wide['run_name'] == pipeline].iloc[0]
    values = [row[m] for m in retriever_metrics]
    bars = ax.bar(x + i * width, values, width,
                  label=LABELS[pipeline], color=PALETTE[pipeline],
                  alpha=0.88, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(ret_labels)
ax.set_ylabel('Score (%)')
ax.set_title('(A) Retriever Metrics')
ax.set_ylim(0, 65)
ax.legend(framealpha=0.9)

# --- Panel B: Generator metrics ---
ax2 = axes[1]
gen_metrics = ['faithfulness', 'hallucination', 'context_utilization']
gen_labels  = ['Faithfulness', 'Hallucination', 'Context\nUtilization']

x2 = np.arange(len(gen_metrics))

for i, pipeline in enumerate(PIPELINES):
    row = df_rc_wide[df_rc_wide['run_name'] == pipeline].iloc[0]
    values = [row[m] for m in gen_metrics]
    bars = ax2.bar(x2 + i * width, values, width,
                   label=LABELS[pipeline], color=PALETTE[pipeline],
                   alpha=0.88, edgecolor='white')
    for bar, val in zip(bars, values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax2.set_xticks(x2 + width)
ax2.set_xticklabels(gen_labels)
ax2.set_ylabel('Score (%)')
ax2.set_title('(B) Generator Metrics')
ax2.set_ylim(0, 58)
ax2.legend(framealpha=0.9)

# Annotate: hallucination is inverted (lower = better)
ax2.annotate('↓ lower is better', xy=(1 + width, df_rc_wide[df_rc_wide['run_name']=='dense'].iloc[0]['hallucination'] - 3),
             fontsize=8, color='gray', ha='center')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig2_ragchecker_metrics.png')
plt.show()
print('Figure 2 saved.')


In [ ]:
# =============================================================================
# Σχήμα 3: Συνολικές μετρικές RAGChecker
# =============================================================================

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('Figure 3: Overall RAGChecker Performance (Precision / Recall / F1)',
             fontsize=13, fontweight='bold')

overall_metrics = ['precision', 'recall', 'f1']
ov_labels = ['Precision', 'Recall', 'F1']
x = np.arange(len(overall_metrics))
width = 0.25

for i, pipeline in enumerate(PIPELINES):
    row = df_rc_wide[df_rc_wide['run_name'] == pipeline].iloc[0]
    values = [row[m] for m in overall_metrics]
    bars = ax.bar(x + i * width, values, width,
                  label=LABELS[pipeline], color=PALETTE[pipeline],
                  alpha=0.88, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(ov_labels)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 45)
ax.legend(framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig3_ragchecker_overall.png')
plt.show()
print('Figure 3 saved.')


In [ ]:
# =============================================================================
# Σχήμα 4: Ντετερμινιστικές μετρικές QA
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
fig.suptitle('Figure 4: QA Deterministic Evaluation Metrics',
             fontsize=13, fontweight='bold', y=1.02)

qa_metrics  = ['avg_lexical_f1', 'exact_substring_match_rate', 'avg_context_support_score']
qa_mlabels  = ['Avg. Lexical F1', 'Exact Substring Match', 'Context Support Score']
qa_scale    = [100, 100, 100]   # convert 0-1 → percentage

for idx, (metric, label) in enumerate(zip(qa_metrics, qa_mlabels)):
    ax = axes[idx]
    values = [
        df_qa[df_qa['run_name'] == p].iloc[0][metric] * qa_scale[idx]
        for p in PIPELINES
    ]
    bars = ax.bar([LABELS[p] for p in PIPELINES], values,
                  color=[PALETTE[p] for p in PIPELINES],
                  alpha=0.88, edgecolor='white', width=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(label)
    ax.set_ylabel('%')
    ax.set_ylim(0, max(values) * 1.3 + 5)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_qa_deterministic.png')
plt.show()
print('Figure 4 saved.')


In [ ]:
# =============================================================================
# Σχήμα 5: Συγκριτικό διάγραμμα radar
# =============================================================================

# Metrics selected for radar — all normalized to 0-100, higher=better
# Note: hallucination is INVERTED (lower hallucination = better)
radar_metrics = {
    'Hit@1'            : 'evidence_hit_at_1',
    'Hit@5'            : 'evidence_hit_at_5',
    'MRR'              : 'mrr',
    'Claim Recall'     : None,   # from RAGChecker
    'Faithfulness'     : None,
    'Anti-Hallucination': None,  # = 100 - hallucination
    'Context Support'  : None,   # from QA
}

# Build values dict
radar_data = {}
for pipeline in PIPELINES:
    r_row = df_retrieval[df_retrieval['run_name'] == pipeline].iloc[0]
    rc_row = df_rc_wide[df_rc_wide['run_name'] == pipeline].iloc[0]
    qa_row = df_qa[df_qa['run_name'] == pipeline].iloc[0]
    
    radar_data[pipeline] = [
        r_row['evidence_hit_at_1'] * 100,
        r_row['evidence_hit_at_5'] * 100,
        r_row['mrr'] * 100,
        rc_row['claim_recall'],
        rc_row['faithfulness'],
        100 - rc_row['hallucination'],   # inverted
        qa_row['avg_context_support_score'] * 100,
    ]

labels = list(radar_metrics.keys())
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
fig.suptitle('Figure 5: Multi-Dimensional Pipeline Comparison (Radar Chart)',
             fontsize=13, fontweight='bold', y=1.02)

for pipeline in PIPELINES:
    values = radar_data[pipeline]
    values += values[:1]  # close
    ax.plot(angles, values, 'o-', linewidth=2,
            label=LABELS[pipeline], color=PALETTE[pipeline])
    ax.fill(angles, values, alpha=0.10, color=PALETTE[pipeline])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 100)
ax.yaxis.set_tick_params(labelsize=8)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20', '40', '60', '80', '100'], color='gray', fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig5_radar_chart.png')
plt.show()
print('Figure 5 saved.')


In [ ]:
# =============================================================================
# Σχήμα 6: Κατανομή RAGChecker F1 ανά ερώτημα
# =============================================================================

# Check column names in details
print(df_rc_details.columns.tolist())
print(df_rc_details.head(2))


In [ ]:
# =============================================================================
# Σχήμα 6: Κατανομή επιδόσεων ανά ερώτημα
# =============================================================================

# Identify numeric per-query metric columns
metric_cols = [c for c in df_rc_details.columns
               if c.startswith('metrics.') and df_rc_details[c].dtype in [np.float64, float]]
print('Metric columns:', metric_cols)

# Focus: f1 per query per pipeline
f1_col = 'metrics.f1' if 'metrics.f1' in df_rc_details.columns else metric_cols[0]
faith_col = 'metrics.faithfulness' if 'metrics.faithfulness' in df_rc_details.columns else None
hall_col  = 'metrics.hallucination' if 'metrics.hallucination' in df_rc_details.columns else None

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Figure 6: Per-Query Score Distributions (RAGChecker)',
             fontsize=13, fontweight='bold', y=1.02)

for ax_idx, (col, title) in enumerate([
        (f1_col, 'F1 Score'),
        (faith_col, 'Faithfulness'),
        (hall_col, 'Hallucination')]):
    ax = axes[ax_idx]
    if col is None:
        ax.set_visible(False)
        continue
    data_groups = [
        df_rc_details[df_rc_details['run_name'] == p][col].dropna().values
        for p in PIPELINES
    ]
    bp = ax.boxplot(data_groups, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.2),
                    flierprops=dict(marker='o', markersize=3, alpha=0.4))
    for patch, pipeline in zip(bp['boxes'], PIPELINES):
        patch.set_facecolor(PALETTE[pipeline])
        patch.set_alpha(0.7)
    
    ax.set_xticks(range(1, len(PIPELINES)+1))
    ax.set_xticklabels([LABELS[p] for p in PIPELINES], rotation=15, ha='right')
    ax.set_title(title)
    ax.set_ylabel('Score (0–1)')
    ax.set_ylim(-0.05, 1.05)
    if col == hall_col:
        ax.set_title(f'{title} ↓ lower is better', fontsize=10)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig6_per_query_distributions.png')
plt.show()
print('Figure 6 saved.')


In [ ]:
# =============================================================================
# Σχήμα 7: Συγκεντρωτικός πίνακας αποτελεσμάτων
# =============================================================================

# Build summary table
rows = []
for pipeline in PIPELINES:
    r_row  = df_retrieval[df_retrieval['run_name'] == pipeline].iloc[0]
    rc_row = df_rc_wide[df_rc_wide['run_name'] == pipeline].iloc[0]
    qa_row = df_qa[df_qa['run_name'] == pipeline].iloc[0]
    rows.append({
        'Pipeline'          : LABELS[pipeline],
        'Hit@1 (%)'         : f"{r_row['evidence_hit_at_1']*100:.1f}",
        'Hit@5 (%)'         : f"{r_row['evidence_hit_at_5']*100:.1f}",
        'MRR (%)'           : f"{r_row['mrr']*100:.1f}",
        'Claim Recall (%)'  : f"{rc_row['claim_recall']:.1f}",
        'Faithfulness (%)'  : f"{rc_row['faithfulness']:.1f}",
        'Hallucination (%)'  : f"{rc_row['hallucination']:.1f}",
        'Ctx. Support (%)'  : f"{qa_row['avg_context_support_score']*100:.1f}",
        'RAGChecker F1 (%)' : f"{rc_row['f1']:.1f}",
    })

df_table = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(15, 2.5))
ax.axis('off')

col_widths = [0.18] + [0.10] * (len(df_table.columns) - 1)

tbl = ax.table(
    cellText=df_table.values,
    colLabels=df_table.columns,
    cellLoc='center',
    loc='center',
    colWidths=col_widths,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 1.6)

# Style header
for j in range(len(df_table.columns)):
    tbl[(0, j)].set_facecolor('#2E4057')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Style rows
row_colors = ['#EBF5FB', '#FDFEFE', '#FDFEFE']
pipeline_colors_light = ['#D6E4F7', '#D5F0E0', '#FAD7D7']
for i, pipeline in enumerate(PIPELINES):
    for j in range(len(df_table.columns)):
        tbl[(i+1, j)].set_facecolor(pipeline_colors_light[i])

# Highlight best values (manually — last row = reranked)
best_cols = [1, 2, 3, 4, 5, 7, 8]  # higher is better
for col_idx in best_cols:
    values = [float(tbl[(r+1, col_idx)].get_text().get_text()) for r in range(3)]
    best_row = np.argmax(values)
    tbl[(best_row+1, col_idx)].set_text_props(fontweight='bold')
    tbl[(best_row+1, col_idx)].set_facecolor('#A9DFBF')

# Hallucination: lower is better (col index 6)
hall_values = [float(tbl[(r+1, 6)].get_text().get_text()) for r in range(3)]
best_hall = np.argmin(hall_values)
tbl[(best_hall+1, 6)].set_text_props(fontweight='bold')
tbl[(best_hall+1, 6)].set_facecolor('#A9DFBF')

fig.suptitle('Table 1: Summary of All Evaluation Metrics per Pipeline (best values highlighted in green)',
             fontsize=11, fontweight='bold', y=1.05)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig7_summary_table.png')
plt.show()
print('Figure 7 saved.')


In [ ]:
# =============================================================================
# Σχήμα 8: Μεταβολή απόδοσης σε σχέση με το dense baseline
# =============================================================================

# Δ values relative to dense baseline
baseline = {
    'Hit@1': df_retrieval[df_retrieval['run_name']=='dense'].iloc[0]['evidence_hit_at_1']*100,
    'Hit@5': df_retrieval[df_retrieval['run_name']=='dense'].iloc[0]['evidence_hit_at_5']*100,
    'MRR'  : df_retrieval[df_retrieval['run_name']=='dense'].iloc[0]['mrr']*100,
    'Claim\nRecall': df_rc_wide[df_rc_wide['run_name']=='dense'].iloc[0]['claim_recall'],
    'Faith.': df_rc_wide[df_rc_wide['run_name']=='dense'].iloc[0]['faithfulness'],
    'Anti-Hall.': 100 - df_rc_wide[df_rc_wide['run_name']=='dense'].iloc[0]['hallucination'],
    'F1'   : df_rc_wide[df_rc_wide['run_name']=='dense'].iloc[0]['f1'],
}

def get_vals(pipeline):
    r  = df_retrieval[df_retrieval['run_name']==pipeline].iloc[0]
    rc = df_rc_wide[df_rc_wide['run_name']==pipeline].iloc[0]
    return {
        'Hit@1': r['evidence_hit_at_1']*100,
        'Hit@5': r['evidence_hit_at_5']*100,
        'MRR'  : r['mrr']*100,
        'Claim\nRecall': rc['claim_recall'],
        'Faith.': rc['faithfulness'],
        'Anti-Hall.': 100 - rc['hallucination'],
        'F1'   : rc['f1'],
    }

metric_keys = list(baseline.keys())
x = np.arange(len(metric_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Figure 8: Percentage-Point Improvement over Dense Baseline',
             fontsize=13, fontweight='bold')

for offset, pipeline in zip([-width/2, width/2], ['hybrid', 'hybrid_reranked']):
    vals = get_vals(pipeline)
    deltas = [vals[k] - baseline[k] for k in metric_keys]
    bars = ax.bar(x + offset, deltas, width,
                  label=LABELS[pipeline], color=PALETTE[pipeline],
                  alpha=0.88, edgecolor='white')
    for bar, d in zip(bars, deltas):
        sign = '+' if d >= 0 else ''
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (0.2 if d >= 0 else -1.2),
                f'{sign}{d:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.axhline(0, color='black', linewidth=1, linestyle='-')
ax.set_xticks(x)
ax.set_xticklabels(metric_keys)
ax.set_ylabel('Δ Percentage Points vs. Dense')
ax.legend(framealpha=0.9)
ax.set_ylim(-5, 20)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig8_delta_improvement.png')
plt.show()
print('Figure 8 saved.')


In [ ]:
# =============================================================================
# Σχήμα 9: Ανάλυση generator με RAGChecker
#          (Noise Sensitivity & Self-Knowledge)
# =============================================================================

gen_full = ['context_utilization', 'faithfulness', 'hallucination',
            'noise_sensitivity_in_relevant', 'noise_sensitivity_in_irrelevant', 'self_knowledge']
gen_full_labels = ['Context\nUtilization', 'Faithfulness', 'Hallucination',
                   'Noise Sens.\n(Relevant)', 'Noise Sens.\n(Irrelevant)', 'Self\nKnowledge']

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Figure 9: RAGChecker Generator Metrics — Full Breakdown',
             fontsize=13, fontweight='bold')

x = np.arange(len(gen_full))
width = 0.25

for i, pipeline in enumerate(PIPELINES):
    row = df_rc_wide[df_rc_wide['run_name'] == pipeline].iloc[0]
    values = [row[m] for m in gen_full]
    bars = ax.bar(x + i * width, values, width,
                  label=LABELS[pipeline], color=PALETTE[pipeline],
                  alpha=0.88, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width)
ax.set_xticklabels(gen_full_labels, fontsize=9.5)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 58)
ax.legend(framealpha=0.9)

# Annotate: which direction is better for each metric
for idx, m in enumerate(gen_full):
    direction = '↓' if m in ['hallucination', 'noise_sensitivity_in_relevant', 'noise_sensitivity_in_irrelevant'] else '↑'
    ax.text(idx + width, -4, direction, ha='center', fontsize=11, color='gray')

ax.set_ylim(-5, 60)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig9_generator_deep_dive.png')
plt.show()
print('Figure 9 saved.')


In [ ]:
# =============================================================================
# Πρόσθετα δεδομένα για συγκριτική ανάλυση
# =============================================================================
working_df = pd.read_csv('../data/interim/financebench_open_source_working.csv')
qa_dense = pd.read_csv('../data/processed/qa_results/rag_qa_results_dense.csv')
qa_hybrid = pd.read_csv('../data/processed/qa_results/rag_qa_results_hybrid.csv')
qa_reranked = pd.read_csv('../data/processed/qa_results/rag_qa_results_hybrid_reranked.csv')
df_error_compare = pd.read_csv('../data/processed/error_analysis/error_cross_run_comparison.csv')
df_examples = pd.read_csv('../data/processed/evaluation/qualitative_examples.csv')

def normalize_text(text):
    text = str(text).lower().strip()
    text = text.replace('$', ' ').replace('%', ' percent ')
    text = ''.join(ch if ch.isalnum() or ch.isspace() else ' ' for ch in text)
    return ' '.join(text.split())

def lexical_f1(pred, gold):
    p = normalize_text(pred).split()
    g = normalize_text(gold).split()
    if not p or not g:
        return 0.0
    common = {tok: min(p.count(tok), g.count(tok)) for tok in set(p) & set(g)}
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def prepare_qa_detail(qa_df, run_name):
    cols = ['financebench_id', 'question', 'expected_answer', 'generated_answer', 'expected_doc_name', 'top_doc_id', 'doc_match']
    df = qa_df[cols].copy()
    df['run_name'] = run_name
    df['lexical_f1'] = [lexical_f1(p, g) for p, g in zip(df['generated_answer'], df['expected_answer'])]
    df['insufficient_evidence'] = df['generated_answer'].astype(str).str.contains('Insufficient evidence', case=False, na=False)
    return df

qa_detail = pd.concat([
    prepare_qa_detail(qa_dense, 'dense'),
    prepare_qa_detail(qa_hybrid, 'hybrid'),
    prepare_qa_detail(qa_reranked, 'hybrid_reranked')
], ignore_index=True)

question_meta = working_df[['financebench_id', 'question_type', 'question_reasoning', 'company', 'doc_type', 'doc_period']].drop_duplicates()
qa_detail = qa_detail.merge(question_meta, on='financebench_id', how='left')

print('Τα πρόσθετα δεδομένα συγκριτικής ανάλυσης προετοιμάστηκαν.')
print(f'   QA detail rows: {qa_detail.shape}')
print(f'   Error comparison rows: {df_error_compare.shape}')
print(f'   Qualitative examples rows: {df_examples.shape}')


In [ ]:
# =============================================================================
# Σχήμα 10: Κατηγοριοποίηση σφαλμάτων ανά pipeline
# =============================================================================
failure_order = ['SUCCESS', 'GOOD_RETRIEVAL_BAD_ANSWER', 'INSUFFICIENT_EVIDENCE', 'WRONG_YEAR_DOC', 'RETRIEVAL_MISS']
failure_colors = {
    'SUCCESS': '#2E8B57',
    'GOOD_RETRIEVAL_BAD_ANSWER': '#E17C05',
    'INSUFFICIENT_EVIDENCE': '#C44E52',
    'WRONG_YEAR_DOC': '#8172B2',
    'RETRIEVAL_MISS': '#4C72B0',
}
df_fail = df_error_compare.copy()
df_fail['percentage'] = df_fail['percentage'].astype(float)
pivot_fail = df_fail.pivot(index='run', columns='failure_category', values='percentage').fillna(0)
pivot_fail = pivot_fail.reindex(PIPELINES)
fig, ax = plt.subplots(figsize=(12, 7))
bottom = np.zeros(len(pivot_fail))
for cat in failure_order:
    vals = pivot_fail[cat].values if cat in pivot_fail.columns else np.zeros(len(pivot_fail))
    ax.bar([LABELS[p] for p in PIPELINES], vals, bottom=bottom, color=failure_colors[cat], label=cat.replace('_', ' ').title())
    for i, v in enumerate(vals):
        if v >= 6:
            ax.text(i, bottom[i] + v / 2, f'{v:.1f}%', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottom += vals
ax.set_ylabel('Queries (%)')
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title('Figure 10: Failure Taxonomy Composition by Pipeline', pad=14, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig10_failure_taxonomy.png')
print('Saved Figure 10.')
plt.show()


In [ ]:
# =============================================================================
# Σχήμα 11: Ροή από το retrieval έως την ποιότητα απάντησης
# =============================================================================
funnel_df = df_retrieval[['run_name', 'evidence_hit_at_1']].merge(
    df_qa[['run_name', 'avg_context_support_score', 'exact_substring_match_rate', 'avg_lexical_f1']], on='run_name', how='left'
)
funnel_df = funnel_df.set_index('run_name').reindex(PIPELINES).reset_index()
stages = ['Top-1 Doc Match', 'Context Support', 'Exact Match', 'Lexical F1']
stage_cols = ['evidence_hit_at_1', 'avg_context_support_score', 'exact_substring_match_rate', 'avg_lexical_f1']
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(stages))
for run in PIPELINES:
    row = funnel_df[funnel_df['run_name'] == run].iloc[0]
    y = [row[c] * 100 for c in stage_cols]
    ax.plot(x, y, marker='o', linewidth=2.8, markersize=8, color=PALETTE[run], label=LABELS[run])
    for xi, yi in zip(x, y):
        ax.text(xi, yi + 1.2, f'{yi:.1f}', ha='center', va='bottom', fontsize=9, color=PALETTE[run])
ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylabel('Score (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylim(0, max(70, ax.get_ylim()[1]))
ax.set_title('Figure 11: Retrieval-to-Answer Performance Funnel', pad=14, fontweight='bold')
ax.legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig11_pipeline_funnel.png')
print('Saved Figure 11.')
plt.show()


In [ ]:
# =============================================================================
# Σχήμα 12: Απόδοση ανά τύπο ερώτησης
# =============================================================================
qt = qa_detail.copy()
qt['doc_match'] = qt['doc_match'].astype(str).str.lower().eq('true').astype(int)
qt['question_type'] = qt['question_type'].fillna('unknown')
question_type_order = qt.groupby('question_type')['financebench_id'].count().sort_values(ascending=False).index.tolist()
qt_metric = qt.groupby(['question_type', 'run_name'])['lexical_f1'].mean().reset_index()
heat = qt_metric.pivot(index='question_type', columns='run_name', values='lexical_f1').reindex(question_type_order)
heat = heat[[p for p in PIPELINES if p in heat.columns]] * 100
fig, ax = plt.subplots(figsize=(10, max(5, 0.55 * len(heat))))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlGnBu', linewidths=0.5, cbar_kws={'label': 'Mean Lexical F1 (%)'}, ax=ax)
ax.set_title('Figure 12: Mean Answer Quality by Question Type', pad=14, fontweight='bold')
ax.set_xlabel('Pipeline')
ax.set_ylabel('Question Type')
ax.set_xticklabels([LABELS.get(c, c) for c in heat.columns], rotation=15, ha='right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig12_question_type_heatmap.png')
print('Saved Figure 12.')
plt.show()


In [ ]:
# =============================================================================
# Σχήμα 13: Άμεσες συγκρίσεις ανά ερώτημα
# =============================================================================
wide_f1 = qa_detail.pivot_table(index='financebench_id', columns='run_name', values='lexical_f1')
pairs = [('dense', 'hybrid'), ('dense', 'hybrid_reranked'), ('hybrid', 'hybrid_reranked')]
rows = []
for a, b in pairs:
    diff = wide_f1[b] - wide_f1[a]
    rows.append({'comparison': f'{LABELS[a]} vs {LABELS[b]}', 'outcome': f'{LABELS[a]} wins', 'count': int((diff < -1e-9).sum())})
    rows.append({'comparison': f'{LABELS[a]} vs {LABELS[b]}', 'outcome': 'Tie', 'count': int((diff.abs() <= 1e-9).sum())})
    rows.append({'comparison': f'{LABELS[a]} vs {LABELS[b]}', 'outcome': f'{LABELS[b]} wins', 'count': int((diff > 1e-9).sum())})
df_wins = pd.DataFrame(rows)
outcome_colors = {
    'Dense (Baseline) wins': PALETTE['dense'],
    'Hybrid wins': PALETTE['hybrid'],
    'Hybrid + Reranking wins': PALETTE['hybrid_reranked'],
    'Tie': '#B0B0B0'
}
fig, ax = plt.subplots(figsize=(12, 7))
comparisons = df_wins['comparison'].unique().tolist()
left = np.zeros(len(comparisons))
for outcome in ['Dense (Baseline) wins', 'Hybrid wins', 'Hybrid + Reranking wins', 'Tie']:
    vals = [df_wins[(df_wins['comparison'] == comp) & (df_wins['outcome'] == outcome)]['count'].sum() for comp in comparisons]
    if any(vals):
        ax.barh(comparisons, vals, left=left, color=outcome_colors[outcome], label=outcome)
        for i, v in enumerate(vals):
            if v >= 8:
                ax.text(left[i] + v / 2, i, str(v), ha='center', va='center', color='white', fontweight='bold', fontsize=9)
        left += vals
ax.set_xlabel('Number of queries')
ax.set_title('Figure 13: Head-to-Head Pipeline Wins per Query (Lexical F1)', pad=14, fontweight='bold')
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig13_head_to_head_wins.png')
print('Saved Figure 13.')
plt.show()


In [ ]:
# =============================================================================
# Πίνακας 2: Αντιπροσωπευτικά ποιοτικά παραδείγματα
# =============================================================================
examples = df_examples.head(5).copy()
display_cols = ['question', 'gold_answer', 'dense_answer', 'hybrid_answer', 'hybrid_reranked_answer']
table_df = examples[display_cols].copy()
rename_map = {
    'question': 'Question',
    'gold_answer': 'Gold',
    'dense_answer': 'Dense',
    'hybrid_answer': 'Hybrid',
    'hybrid_reranked_answer': 'Hybrid + Reranking'
}
table_df = table_df.rename(columns=rename_map)
for col in table_df.columns:
    table_df[col] = table_df[col].astype(str).str.replace('Insufficient evidence in the retrieved context.', 'Insufficient evidence', regex=False).str.slice(0, 140)
fig, ax = plt.subplots(figsize=(22, 8))
ax.axis('off')
tbl = ax.table(cellText=table_df.values, colLabels=table_df.columns, cellLoc='left', colLoc='left', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 2.0)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2F4F4F')
    else:
        cell.set_facecolor('#F7F7F7' if row % 2 == 1 else 'white')
fig.suptitle('??Figure 2: Representative Qualitative Examples Across Pipelines', y=0.98, fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig14_qualitative_examples_table.png')
print('Saved ??Figure 2 figure.')
plt.show()


In [ ]:
# =============================================================================
# Εκτεταμένο ευρετήριο σχημάτων
# =============================================================================
extended_descriptions = {
    'fig10_failure_taxonomy.png': 'Figure 10 - Failure taxonomy by pipeline',
    'fig11_pipeline_funnel.png': 'Figure 11 - Retrieval to answer funnel',
    'fig12_question_type_heatmap.png': 'Figure 12 - Mean lexical F1 by question type',
    'fig13_head_to_head_wins.png': 'Figure 13 - Head-to-head wins per query',
    'fig14_qualitative_examples_table.png': 'Table 2 - Representative qualitative examples',
}
print('Εκτεταμένο ευρετήριο σχημάτων:')
for f, desc in extended_descriptions.items():
    print(f'  - {desc}')


In [ ]:
# =============================================================================
# Σύνοψη παραγόμενων σχημάτων
# =============================================================================

figures = sorted([f for f in os.listdir(FIG_DIR) if f.endswith('png')])
print(f'\nΗ παραγωγή σχημάτων ολοκληρώθηκε. Αποθηκεύτηκαν {len(figures)} αρχεία στο ./{FIG_DIR}/')
print('\nΕυρετήριο σχημάτων για τη διπλωματική:')
descriptions = {
    'fig1_retrieval_performance.png'     : 'Σχήμα 1: Hit@k και MRR ανά pipeline',
    'fig2_ragchecker_metrics.png'        : 'Σχήμα 2: Μετρικές retriever και generator από το RAGChecker',
    'fig3_ragchecker_overall.png'        : 'Σχήμα 3: Συνολικές μετρικές F1/Precision/Recall από το RAGChecker',
    'fig4_qa_deterministic.png'          : 'Σχήμα 4: Ντετερμινιστικές μετρικές QA',
    'fig5_radar_chart.png'               : 'Σχήμα 5: Διάγραμμα radar για πολυδιάστατη σύγκριση',
    'fig6_per_query_distributions.png'   : 'Σχήμα 6: Κατανομές βαθμολογιών ανά ερώτημα',
    'fig7_summary_table.png'             : 'Πίνακας 1: Συγκεντρωτικός πίνακας αποτελεσμάτων',
    'fig8_delta_improvement.png'         : 'Σχήμα 8: Μεταβολή απόδοσης σε σχέση με το dense baseline',
    'fig9_generator_deep_dive.png'       : 'Σχήμα 9: Αναλυτική αποτύπωση generator',
    'fig10_failure_taxonomy.png'         : 'Σχήμα 10: Κατηγοριοποίηση σφαλμάτων ανά pipeline',
    'fig11_pipeline_funnel.png'          : 'Σχήμα 11: Ροή από retrieval σε απάντηση',
    'fig12_question_type_heatmap.png'    : 'Σχήμα 12: Μέσο lexical F1 ανά τύπο ερώτησης',
    'fig13_head_to_head_wins.png'        : 'Σχήμα 13: Άμεσες συγκρίσεις ανά ερώτημα',
    'fig14_qualitative_examples_table.png': 'Πίνακας 2: Αντιπροσωπευτικά ποιοτικά παραδείγματα',
}
for f in figures:
    print(f'  - {descriptions.get(f, f)}')
